# Train tiếp YOLO26m ở 1280 trên Kaggle
Hết quota Colab thì chuyển sang đây train tiếp. Notebook này chỉ giữ các bước cần cho việc đó để chạy nền bằng Save Version, Save & Run All. Notebook gốc có cell `google.colab` sẽ báo lỗi trên Kaggle, và có cell test chỉ được chạy một lần.

Cài đặt: GPU T4, bật Internet, Input là dataset có `acne04v2_yolo/` và `acne_yolo26/runs/`.

Chạy lần lượt K1 đến K5. K2 sẽ dừng ngay nếu không thấy run cũ.
Xong thì tải `y26m_1280_run.zip` ở tab Output, các bước đánh giá (mục 3 trở đi) làm ở `acne_yolo26m_1280.ipynb`.


In [ ]:
# Cell K1: cài đặt, tìm dữ liệu, chép run cũ (thay cho S1 và S2 của Colab)
import os, sys, json, time, pickle, hashlib, subprocess, shutil, importlib.util
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
ULTRA_VER = '8.4.192'
try: _v = version('ultralytics')
except PackageNotFoundError: _v = None
if _v != ULTRA_VER:  # kaggle có thể có sẵn bản khác, ép về bản đã train
    print(f'ultralytics hiện có: {_v}, cài lại {ULTRA_VER}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'ultralytics=={ULTRA_VER}'], check=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, ultralytics, yaml
from ultralytics import YOLO
print('ultralytics', ultralytics.__version__, '|', torch.cuda.get_device_name(0))

DRIVE_ROOT = Path('/kaggle/working/acne_yolo26')
RUNS = DRIVE_ROOT / 'runs'; EVAL = DRIVE_ROOT / 'eval'
for p in (RUNS, EVAL): p.mkdir(parents=True, exist_ok=True)
IMGSZ = 1280; RUN_NAME = f'y26m_{IMGSZ}'; SEED = 42
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLIT_DIR = {'train': 'train', 'val': 'valid', 'test': 'test'}
SESSION_T0 = time.time(); SESSION_HOURS = 11.0  # kaggle cắt ở 12h

def find_data_root(base):
    for p in [base, *sorted(base.rglob('*'))]:
        if p.is_dir() and (p / 'train' / 'images').is_dir() and (p / 'valid' / 'images').is_dir():
            return p

DATA_ROOT = find_data_root(Path('/kaggle/input'))
assert DATA_ROOT, 'Chưa Add Input dataset acne04v2_yolo'
DATA_YAML = '/kaggle/working/acne_data_fixed.yaml'
orig_yaml = list(DATA_ROOT.glob('*.yaml'))
names = yaml.safe_load(open(orig_yaml[0])).get('names', ['acne']) if orig_yaml else ['acne']
yaml.safe_dump(dict(path=str(DATA_ROOT), train='train/images', val='valid/images', test='test/images', names=names),
               open(DATA_YAML, 'w'), allow_unicode=True)
print('DATA_ROOT =', DATA_ROOT)

# chép run từ colab sang chỗ ghi đươc, /kaggle/input chỉ đọc
for last in Path('/kaggle/input').rglob('weights/last.pt'):
    dst = RUNS / last.parent.parent.name
    if not dst.exists():
        shutil.copytree(last.parent.parent, dst); print('Đã chép run:', dst.name)

In [ ]:
# Cell K2: ko thấy run cũ thì dừng luôn, khỏi train lại từ đầu
last = RUNS / RUN_NAME / 'weights' / 'last.pt'
assert last.exists(), f'Không thấy {last}'
ck = torch.load(last, map_location='cpu', weights_only=False)
print(f"OK: run {RUN_NAME} | epoch đã xong = {ck['epoch'] + 1}/{ck['train_args']['epochs']} | cache = {ck['train_args'].get('cache')}")
del ck
print('GPU:', torch.cuda.get_device_name(0), '| số GPU:', torch.cuda.device_count())
print(subprocess.run(['free', '-h'], capture_output=True, text=True).stdout)

In [ ]:
# Cell K3: chép từ S3 sang
from PIL import Image
from ultralytics.data.utils import exif_size

# thang Hayashi, đếm trên nửa mặt. Gộp nặng với rất nặng nên còn 3 mức
SEV_BINS = [5, 20]
SEV_NAMES = ['nhẹ', 'vừa', 'nặng']
SEV_ASCII = ['nhe', 'vua', 'nang']  # cv2.putText ko vẽ đươc dấu tiếng việt
def severity(counts):
    return np.digitize(np.asarray(counts), np.array(SEV_BINS), right=True)   # <=5 là 0, 6-20 là 1, >20 là 2

def list_images(split):
    d = DATA_ROOT / SPLIT_DIR[split] / 'images'
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXT)

def label_path(img):
    return Path(img).parent.parent / 'labels' / (Path(img).stem + '.txt')

def read_labels(img):
    """(N,5) cls xc yc w h chuẩn hoá; None nếu thiếu file nhãn."""
    lp = label_path(img)
    if not lp.exists(): return None
    rows = [[float(x) for x in ln.split()] for ln in lp.read_text().strip().splitlines() if len(ln.split()) == 5]
    return np.array(rows, dtype=float).reshape(-1, 5)

def gt_xyxy(img, shape_hw):
    lab = read_labels(img)
    if lab is None or len(lab) == 0: return np.zeros((0, 4))
    h, w = shape_hw[:2]
    xc, yc, bw, bh = lab[:, 1] * w, lab[:, 2] * h, lab[:, 3] * w, lab[:, 4] * h
    return np.stack([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2], 1)

def box_iou(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2]); br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def greedy_match(gt, pr, conf, iou_thr=0.5):
    """Ghép theo conf giảm dần như lúc tính AP, gconf[g] = 0 nếu nhãn g không ghép được."""
    tp = np.zeros(len(pr), bool); gconf = np.zeros(len(gt)); used = np.zeros(len(gt), bool)
    if len(gt) and len(pr):
        order = np.argsort(-conf); ious = box_iou(pr[order], gt)
        for k, j in enumerate(order):
            row = np.where(used, -1.0, ious[k]); g = int(row.argmax())
            if row[g] >= iou_thr:
                tp[j] = True; used[g] = True; gconf[g] = conf[j]
    return tp, gconf

def collect_preds(weights, split='val', imgsz=IMGSZ, end2end=True, iou=0.7, max_det=500, augment=False, batch=8):
    """Dự đoán một lần ở conf rất thấp rồi cache lại để quét ngưỡng sau."""
    w = Path(weights)
    tag = f"{w.parent.parent.name}_{w.stem}_{split}_{imgsz}_e2e{int(end2end)}_iou{iou}_md{max_det}_aug{int(augment)}"
    out = EVAL / 'preds' / f'{tag}.pkl'; out.parent.mkdir(exist_ok=True)
    if out.exists(): return pickle.load(open(out, 'rb'))
    model = YOLO(str(w)); imgs = list_images(split); recs = []
    for i in range(0, len(imgs), batch):
        chunk = [str(p) for p in imgs[i:i + batch]]
        res = model.predict(chunk, imgsz=imgsz, conf=0.01, iou=iou, max_det=max_det, end2end=end2end,
                            augment=augment, half=True, verbose=False)
        for p, r in zip(chunk, res):
            recs.append(dict(img=p, shape=r.orig_shape,
                             xyxy=r.boxes.xyxy.cpu().numpy(), conf=r.boxes.conf.cpu().numpy()))
    pickle.dump(recs, open(out, 'wb'))
    return recs

def prepare(recs, size_ref=640):
    """Ghép nhãn với dự đoán từng ảnh, gside là cạnh ngắn của nhãn quy về 640."""
    per = []
    for r in recs:
        gt = gt_xyxy(r['img'], r['shape'])
        tp, gconf = greedy_match(gt, r['xyxy'], r['conf'])
        s = size_ref / max(r['shape'][:2])
        gside = np.minimum(gt[:, 2] - gt[:, 0], gt[:, 3] - gt[:, 1]) * s if len(gt) else np.zeros(0)
        per.append(dict(img=r['img'], gt=gt, pr=r['xyxy'], conf=r['conf'], tp=tp, gconf=gconf, gside=gside))
    return per

CONF_GRID = np.round(np.arange(0.02, 0.61, 0.01), 2)

def sweep(per, grid=CONF_GRID):
    n_gt = np.array([len(p['gt']) for p in per]); rows = []
    for t in grid:
        cnt = np.array([(p['conf'] >= t).sum() for p in per])
        tp = np.array([(p['tp'] & (p['conf'] >= t)).sum() for p in per])
        e = cnt - n_gt; R = tp.sum() / max(n_gt.sum(), 1); P = tp.sum() / max(cnt.sum(), 1)
        rows.append(dict(conf=t, MAE=np.abs(e).mean(), RMSE=np.sqrt((e ** 2).mean()), bias=e.mean(),
                         recall=R, precision=P, F1=2 * P * R / (P + R + 1e-9),
                         sev_acc=(severity(cnt) == severity(n_gt)).mean(), max_pred=int(cnt.max())))
    return pd.DataFrame(rows)

def best_row(df):
    return df.loc[df.MAE.idxmin()]

def per_image(per, t):
    n_gt = np.array([len(p['gt']) for p in per]); cnt = np.array([(p['conf'] >= t).sum() for p in per])
    tp = np.array([(p['tp'] & (p['conf'] >= t)).sum() for p in per])
    return pd.DataFrame(dict(img=[Path(p['img']).name for p in per], n_gt=n_gt, n_pred=cnt,
                             TP=tp, FP=cnt - tp, FN=n_gt - tp, err=cnt - n_gt))

SIZE_EDGES = [8, 16, 32]; SIZE_NAMES = ['<8px', '8-16px', '16-32px', '>32px']   # px quy về 640
def recall_by_size(per, t):
    side = np.concatenate([p['gside'] for p in per]); hit = np.concatenate([p['gconf'] >= t for p in per])
    b = np.digitize(side, SIZE_EDGES)
    return pd.DataFrame([dict(size=SIZE_NAMES[i], n_gt=int((b == i).sum()),
                              recall=hit[b == i].mean() if (b == i).any() else np.nan) for i in range(4)])

# Cấu hình train, lý do xem trong báo cáo
BASE_ARGS = dict(
    data=DATA_YAML, seed=SEED, deterministic=True, project=str(RUNS), exist_ok=True,
    # cố định AdamW vì 'auto' sẽ đổi sang MuSGD khi quá 10k iteration
    optimizer='AdamW', lr0=0.002, momentum=0.9, weight_decay=0.0005,
    lrf=0.01, cos_lr=True,
    warmup_epochs=3, warmup_bias_lr=0.0,
    hsv_h=0.005,  # màu đỏ là dấu hiệu viêm nên ít xoay hue thôi
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.3,  # mun đã nhỏ sẵn, đừng thu thêm
    shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, mixup=0.0, copy_paste=0.0,
    max_det=500,  # có ảnh 434 mụn, mặc định 300 ko đủ
    cache='ram', workers=2, amp=True, plots=True,
)

# Train có resume và tự dừng khi sắp hết giờ
class PauseTraining(Exception): pass

def retarget_ckpt(ckpt_path, project, data_yaml):
    """Sửa đường dẫn trong checkpoint để resume được khi đổi giữa Colab và Kaggle."""
    ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    a = ck.get('train_args', {})
    new_dir = str(Path(project) / Path(ckpt_path).parent.parent.name)
    if a.get('data') == str(data_yaml) and a.get('save_dir') == new_dir: return
    a.update(data=str(data_yaml), project=str(project), save_dir=new_dir)
    ck['train_args'] = a; torch.save(ck, ckpt_path)
    print('Đã sửa đường dẫn trong', ckpt_path)

def hours_left():
    return SESSION_HOURS - (time.time() - SESSION_T0) / 3600

def run_train(name, weights='yolo26m.pt', stop_epoch=None, **overrides):
    """Train có resume, tự dừng khi hết SESSION_HOURS. Xong trả save_dir, tạm dừng trả None."""
    save_dir = RUNS / name; done = save_dir / 'DONE.json'; last = save_dir / 'weights' / 'last.pt'
    if done.exists():
        print(f'{name} đã xong, bỏ qua.'); return save_dir
    if hours_left() < 0.25:
        print(f'Phiên chỉ còn {hours_left():.2f} giờ, chưa bắt đầu {name}. Phiên sau chạy lại cell.'); return None
    args = {**BASE_ARGS, **overrides, 'name': name}
    def pause_cb(tr):  # chạy sau khi last.pt đã lưu
        ep = tr.epoch + 1
        if ep >= tr.epochs: return  # epoch cuối thì để ultralytics tự kết thúc
        if hours_left() < 0.15 or (stop_epoch and ep >= stop_epoch):
            raise PauseTraining(f'{name}: dừng sau epoch {ep}/{tr.epochs}. last.pt đã lưu, phiên sau chạy lại cell để train tiếp.')
    try:
        if last.exists():
            retarget_ckpt(last, RUNS, DATA_YAML)
            model = YOLO(str(last)); model.add_callback('on_fit_epoch_end', pause_cb)
            print(f'Resume {name} (phiên còn {hours_left():.1f} giờ)')
            try:
                model.train(resume=True)
            except PauseTraining:
                raise
            except Exception as e:
                msg = str(e).lower()
                if 'finished' in msg or 'nothing to resume' in msg: print('Run đã kết thúc trước đó.')
                else: raise
        else:
            model = YOLO(weights); model.add_callback('on_fit_epoch_end', pause_cb)
            print(f'Bắt đầu {name} (phiên còn {hours_left():.1f} giờ)')
            model.train(**args)
    except PauseTraining as e:
        print(e); return None
    json.dump(dict(finished=time.strftime('%Y-%m-%d %H:%M'), args=args), open(done, 'w'), indent=1, default=str)
    return save_dir

def pack_run(name, out_dir):
    """Nén run thành <out_dir>/<name>_run.zip."""
    z = shutil.make_archive(str(Path(out_dir) / f'{name}_run'), 'zip', root_dir=RUNS, base_dir=name)
    print('Đã tạo', z, f'({Path(z).stat().st_size / 1e6:.0f} MB)'); return z

print(f'Đã nạp hàm tiện ích | IMGSZ={IMGSZ} | RUN_NAME={RUN_NAME} | phiên còn {hours_left():.1f} giờ')

In [ ]:
# Cell K4: train tiếp đến epoch 100, dừng trước giới hạn 12 tiếng
TRAIN = dict(imgsz=IMGSZ, batch=4, epochs=100, patience=30, close_mosaic=15, save_period=20)
run_train(RUN_NAME, **TRAIN)

In [ ]:
# Cell K5: nén run để tải về (tab Output, file y26m_1280_run.zip)
pack_run(RUN_NAME, '/kaggle/working')